<font size="6" color='grey'><b>KI-Agenten. Verstehen. Anwenden. Gestalten.</b></font>

<font size="5" color='grey'><b>Projekt-Template C – Support-Team</b></font>

---

> **Vorlage für M23 Multi-Agent-Projekt**
> Suche relevante `# ⚠️ TODO`-Kommentare und passe sie an dein Thema an.

**Team-Struktur:**

| Rolle | Agent | Aufgabe |
|-------|-------|---------|
| 🎯 Koordination | **Supervisor** | Klassifikation & Routing |
| 🙋 Standardfragen | **FAQ-Agent** | Häufige Fragen schnell beantworten |
| 🧠 Spezialist | **Specialist** | Komplexe Anfragen bearbeiten |

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
# ⚠️ TODO: Passe den Projektnamen an deinen Team-Namen an
os.environ["LANGSMITH_PROJECT"]  = "M21-Support-Team"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid, show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

## 1 | Übersicht & Lernziel
---

Dieses Template implementiert ein **Support-Team** bestehend aus:
- **FAQ-Agent:** Beantwortet Standardfragen schnell aus einer Wissensbasis
- **Specialist:** Bearbeitet komplexe, individuelle Anfragen
- **Supervisor (Dispatcher):** Klassifiziert die Anfrage und routet zum passenden Agenten

**Routing-Logik:**
```
Einfache Frage  → FAQ-Agent   → FINISH
Komplexe Frage  → Specialist  → FINISH
Grenzfall       → FAQ-Agent   → Specialist → FINISH  (Eskalation)
```

> Dieses Template zeigt **intelligentes Routing**: Der Supervisor entscheidet anhand der Anfrage, nicht einer festen Reihenfolge.

In [ ]:
#@title
#@markdown <p><font size="4" color='green'>flowchart: Support-Team-Architektur</font></p>
diagram = '''
flowchart TD
    START([START]) --> SUP["🎯 Supervisor\nDispatcher"]

    SUP -->|"faq"| F["🙋 FAQ-Agent\nStandard-Antworten"]
    SUP -->|"specialist"| S["🧠 Specialist\nKomplexe Anfragen"]
    SUP -->|"FINISH"| END([END])

    F -->|"AIMessage\n(name=FAQ)"| SUP
    S -->|"AIMessage\n(name=Specialist)"| SUP

    F --- FT1["📚 lookup_faq"]
    F --- FT2["📋 list_categories"]
    S --- ST1["🔍 analyze_problem"]
    S --- ST2["💡 propose_solution"]

    note["🔀 Routing:\nEinfach → FAQ\nKomplex → Specialist\nEskalation: FAQ → Specialist"]

    style SUP  fill:#FF9800,color:#fff
    style F    fill:#2196F3,color:#fff
    style S    fill:#9C27B0,color:#fff
    style FT1  fill:#E3F2FD
    style FT2  fill:#E3F2FD
    style ST1  fill:#F3E5F5
    style ST2  fill:#F3E5F5
    style note fill:#FFF9C4
'''
mermaid(diagram, width=800)

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import create_react_agent
from IPython.display import Image as IPImage

In [ ]:
# Worker-LLM: gpt-4o-mini (Inhalte generieren, schnell & günstig)
worker_llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)

# Supervisor-LLM: o3 (Routing-Entscheidungen, starkes Reasoning)
# ⚠️ o3 akzeptiert KEINEN temperature-Parameter (API-Fehler)
from langchain_core.messages import SystemMessage as SM
import langchain
_base_supervisor = init_chat_model("openai:o3")
print("✅ LLMs initialisiert (worker: gpt-4o-mini | supervisor: o3)")

In [ ]:
# ── FAQ-Agent Tools ────────────────────────────────────────────────────────

# ⚠️ TODO: Erweitere die FAQ_DATABASE mit echten Fragen aus deinem Themengebiet
FAQ_DATABASE = {
    "preis":        "Unsere Preise beginnen bei 9,99 € / Monat. Details: pricing-page.",
    "passwort":     "Passwort zurücksetzen: Einstellungen → Sicherheit → Passwort ändern.",
    "kündigung":    "Kündigung jederzeit zum Monatsende möglich: Einstellungen → Abonnement.",
    "kontakt":      "Support erreichbar: support@beispiel.de | Mo–Fr 9–17 Uhr.",
    "lieferzeit":   "Standard-Lieferzeit: 3–5 Werktage. Express: 1–2 Werktage.",
    "rückgabe":     "30 Tage Rückgaberecht. Kontaktiere uns für ein Rücksende-Label.",
}

@tool
def lookup_faq(stichwort: str) -> str:
    """Sucht eine Antwort in der FAQ-Datenbank.
    Gibt die Antwort zurück oder 'Nicht gefunden' wenn kein Eintrag existiert.
    """
    stichwort_lower = stichwort.lower()
    for key, antwort in FAQ_DATABASE.items():
        if key in stichwort_lower or stichwort_lower in key:
            return f"FAQ-Antwort: {antwort}"
    return (
        f"Kein FAQ-Eintrag für '{stichwort}' gefunden. "
        "Bitte an Specialist weiterleiten oder allgemeine Hilfe anbieten."
    )

@tool
def list_categories() -> str:
    """Listet alle verfügbaren FAQ-Kategorien auf.
    Nützlich um dem Nutzer Orientierung zu geben.
    """
    kategorien = list(FAQ_DATABASE.keys())
    return f"Verfügbare FAQ-Kategorien: {', '.join(kategorien)}"

# ── Specialist Tools ───────────────────────────────────────────────────────

@tool
def analyze_problem(beschreibung: str) -> str:
    """Analysiert eine komplexe Anfrage und strukturiert das Problem.
    Nutze als ersten Schritt bei unklaren oder mehrteiligen Anfragen.
    """
    return (
        f"Problem-Analyse für: '{beschreibung}'\n"
        "Erkannte Aspekte:\n"
        "  1. Hauptproblem: (bitte konkretisieren)\n"
        "  2. Betroffener Bereich: (bitte einordnen)\n"
        "  3. Dringlichkeit: (hoch/mittel/niedrig)\n"
        "  4. Benötigte Informationen: (bitte angeben)"
    )

@tool
def propose_solution(problem: str) -> str:
    """Erarbeitet eine Lösungsstrategie für ein analysiertes Problem.
    Nutze nach analyze_problem für konkrete Handlungsempfehlungen.
    """
    return (
        f"Lösungsvorschlag für '{problem}':\n"
        "Schritt 1: Sofortmaßnahme – (was jetzt?)\n"
        "Schritt 2: Ursachenanalyse – (warum ist es passiert?)\n"
        "Schritt 3: Langfristige Lösung – (wie vermeiden?)\n"
        "Eskalation an: (bei Bedarf welche Abteilung/Person?)"
    )

print("✅ Tools definiert: lookup_faq, list_categories, analyze_problem, propose_solution")

In [ ]:
# ── Worker Agents ──────────────────────────────────────────────────────────

# ⚠️ TODO: Passe die System-Prompts an dein Support-Themengebiet an

faq_agent = create_react_agent(
    worker_llm,
    tools=[lookup_faq, list_categories],
    prompt=SystemMessage(
        # ⚠️ TODO: Definiere das Themengebiet des FAQ-Agenten
        "Du bist ein freundlicher FAQ-Agent. "
        "Nutze lookup_faq um Standardfragen schnell zu beantworten. "
        "Bei 'Nicht gefunden': Nutze list_categories und weise höflich darauf hin, "
        "dass du die Frage an einen Spezialisten weiterleitest. "
        "Antworte kurz, klar und hilfreich. Antworte auf Deutsch."
    )
)

specialist_agent = create_react_agent(
    worker_llm,
    tools=[analyze_problem, propose_solution],
    prompt=SystemMessage(
        # ⚠️ TODO: Definiere das Fachgebiet des Specialists
        "Du bist ein erfahrener Support-Spezialist für komplexe Anfragen. "
        "Nutze analyze_problem um das Problem zu verstehen und "
        "propose_solution für konkrete Lösungsschritte. "
        "Sei empathisch, präzise und lösungsorientiert. Antworte auf Deutsch."
    )
)

print("✅ Worker Agents erstellt:")
print("   faq_agent        → [lookup_faq, list_categories]")
print("   specialist_agent → [analyze_problem, propose_solution]")

In [ ]:
# ── State Definition ───────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages:    Annotated[list, add_messages]  # Gesamte Nachrichten-Historie
    aufgabe:     str    # Original-Aufgabe (unveränderlich)
    naechster:   str    # Routing-Entscheidung des Supervisors
    iterationen: int    # Zähler gegen Endlosschleifen
    max_iter:    int    # Konfigurierbare Grenze (Standard: 6)

# ── Routing-Entscheidung mit Begründung ────────────────────────────────────
class SupervisorEntscheidung(BaseModel):
    """Strukturierte Routing-Entscheidung des Supervisors."""
    naechster: Literal["faq", "specialist", "FINISH"] = Field(
        description="Nächster Agent oder FINISH wenn die Aufgabe vollständig ist."
    )
    begruendung: str = Field(
        description="Kurze Begründung für diese Entscheidung (1–2 Sätze)."
    )

supervisor_llm = (
    _base_supervisor
    .with_structured_output(SupervisorEntscheidung)
    .with_retry(stop_after_attempt=3)
)
print("✅ State und Supervisor-LLM konfiguriert")

In [ ]:
SUPERVISOR_PROMPT = (
    "Du bist Support-Dispatcher. Klassifiziere Anfragen und route sie optimal.\n\n"
    "<Team>\n"
    "- faq:        Für Standardfragen (Preis, Passwort, Kündigung, Lieferzeit)\n"
    "- specialist: Für komplexe, individuelle oder technische Probleme\n"
    "</Team>\n\n"
    # ⚠️ TODO: Passe die Routing-Regeln an dein Support-Thema an
    "<Routing-Regeln>\n"
    "→ faq:        Klare Standardfrage mit bekannter Antwort\n"
    "→ specialist: Komplexes Problem, Beschwerde, technischer Fehler, Sonderfall\n"
    "→ Eskalation: faq hat 'nicht gefunden' → specialist aufrufen\n"
    "→ FINISH:     Frage vollständig beantwortet\n"
    "</Routing-Regeln>\n\n"
    "<Rules>\n"
    "1. Immer zuerst faq versuchen – schnell und günstig.\n"
    "2. specialist nur bei Bedarf (Eskalation oder komplexe Anfrage).\n"
    "3. FINISH nach der ersten vollständigen Antwort.\n"
    "4. Jeden Agenten maximal EINMAL aufrufen.\n"
    "</Rules>"
)
print("✅ Supervisor-Prompt (Dispatcher) konfiguriert")

In [ ]:
# ── Supervisor Node ────────────────────────────────────────────────────────
def supervisor_node(state: AgentState) -> dict:
    """Liest Situation, entscheidet wer als Nächstes arbeitet."""
    # Iterations-Schutz: verhindert Endlosschleifen
    if state["iterationen"] >= state["max_iter"]:
        mprint(f"⚠️ **Iterations-Limit ({state['max_iter']}) erreicht – FINISH erzwungen.**")
        return {"naechster": "FINISH"}

    entscheidung = supervisor_llm.invoke([
        SystemMessage(SUPERVISOR_PROMPT),
        HumanMessage(f"Aufgabe: {state['aufgabe']}"),
        *state["messages"],
    ])
    mprint(f"**👔 Supervisor → `{entscheidung.naechster}`** – _{entscheidung.begruendung}_")
    return {"naechster": entscheidung.naechster}


# ── Worker Node Factory ────────────────────────────────────────────────────
def make_worker_node(agent, name: str):
    """Erzeugt einen Worker-Node mit Fehlerbehandlung und Kontext-Übergabe."""
    def worker_node(state: AgentState) -> dict:
        mprint(f"\n**🤖 {name}** arbeitet...")
        # Aufgabe + bisherige Ergebnisse als Kontext übergeben
        kontext = f"Aufgabe: {state['aufgabe']}"
        if state["messages"]:
            bisherige = "\n".join(
                f"{getattr(m, 'name', 'User')}: {m.content[:300]}"
                for m in state["messages"]
            )
            kontext += f"\n\nBisherige Arbeit des Teams:\n{bisherige}"

        try:
            result = agent.invoke(
                {"messages": [HumanMessage(kontext)]},
                config={"recursion_limit": 15}
            )
            antwort = result["messages"][-1].content
        except Exception as e:
            # ✅ MVP-Anforderung: Mind. ein Fehlerpfad abgefangen
            antwort = f"❌ Fehler: {e} – bitte alternativen Weg wählen."
            mprint(f"  ⚠️ {name} fehlgeschlagen: {e}")

        return {
            "messages":    [AIMessage(content=antwort, name=name)],
            "iterationen": state["iterationen"] + 1,
        }
    worker_node.__name__ = f"{name}_node"
    return worker_node


# ── Router ─────────────────────────────────────────────────────────────────
def supervisor_router(state: AgentState) -> str:
    naechster = state.get("naechster", "FINISH")
    return "__end__" if naechster == "FINISH" else naechster

print("✅ Supervisor-Node, Worker-Nodes und Router definiert")

In [ ]:
# ── Graph aufbauen ────────────────────────────────────────────────────────
builder = StateGraph(AgentState)

# Nodes
builder.add_node("supervisor", supervisor_node)
builder.add_node("faq",        make_worker_node(faq_agent,        "FAQ"))
builder.add_node("specialist", make_worker_node(specialist_agent, "Specialist"))

# Einstiegspunkt
builder.add_edge(START, "supervisor")

# Bedingtes Routing
builder.add_conditional_edges(
    "supervisor",
    supervisor_router,
    {
        "faq":        "faq",
        "specialist": "specialist",
        "__end__": END,
    }
)

# Zurück zum Supervisor nach jedem Worker
builder.add_edge("faq",        "supervisor")
builder.add_edge("specialist", "supervisor")

graph = builder.compile()
print("✅ Graph kompiliert")

# Visualisierung
try:
    display(IPImage(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"(Visualisierung nicht verfügbar: {e})")

In [ ]:
# ── Testlauf 1: Standard-Anfrage (→ FAQ-Agent erwartet) ───────────────────
anfrage1 = "Was kostet das Produkt und wie lange ist die Lieferzeit?"

result1 = graph.invoke(
    {
        "messages":    [],
        "aufgabe":     anfrage1,
        "naechster":   "",
        "iterationen": 0,
        "max_iter":    6,
    },
    config={
        "run_name": "M21-Support-FAQ-Test",
        "tags":     ["m23-support"],
        "recursion_limit": 40,
    }
)

mprint("## 🎯 Test 1: Standard-Anfrage")
mprint(f"**Anfrage:** {anfrage1}\n")
letzte = result1["messages"][-1]
agent_name = getattr(letzte, "name", "Agent")
mprint(f"**[{agent_name}]** {letzte.content}")

# ── Testlauf 2: Komplexe Anfrage (→ Specialist erwartet) ──────────────────
anfrage2 = (
    "Ich habe seit 3 Tagen keinen Zugriff auf mein Konto, "
    "der Passwort-Reset-Link kommt nicht an und meine Bestellung wurde "
    "doppelt abgebucht. Was kann ich tun?"
)

result2 = graph.invoke(
    {
        "messages":    [],
        "aufgabe":     anfrage2,
        "naechster":   "",
        "iterationen": 0,
        "max_iter":    6,
    },
    config={
        "run_name": "M21-Support-Specialist-Test",
        "tags":     ["m23-support"],
        "recursion_limit": 40,
    }
)

mprint("\n## 🎯 Test 2: Komplexe Anfrage")
mprint(f"**Anfrage:** {anfrage2}\n")
letzte2 = result2["messages"][-1]
agent_name2 = getattr(letzte2, "name", "Agent")
mprint(f"**[{agent_name2}]** {letzte2.content}")

In [ ]:
#@title 📊 LangSmith Trace{ display-mode: "form" }
import time; time.sleep(2)
show_trace("M21-Support-Team", limit=2, show_steps=True)

## ✅ MVP-Checkliste (vor Abgabe prüfen)

Dein Projekt ist **fertig**, wenn alle Punkte erfüllt sind:

| Kriterium | Geprüft? |
|-----------|----------|
| Supervisor routet zu **mindestens 2 Workern** | ☐ |
| Worker geben **sinnvolle Antworten** | ☐ |
| Mind. **ein Fehlerpfad** ist abgefangen (bereits im Template ✅) | ☐ |
| **End-to-End-Flow** funktioniert (1 Durchlauf ohne Fehler) | ☐ |
| **LangSmith-Trace** zeigt den Ablauf | ☐ |

**Nicht erforderlich:**
- ❌ Perfekte Ausgabe
- ❌ Schöne UI
- ❌ Mehr als 2 Worker

## A | Deine Aufgabe – Support-Team anpassen

---

### Was du bauen sollst

Du nimmst dieses Template und entwickelst daraus ein **konkretes Support-System** für eine selbst gewählte Domäne. Das System soll einfache Anfragen schnell beantworten (FAQ-Agent) und komplexe Fälle an einen Spezialisten weiterleiten – mit intelligentem Routing durch den Supervisor.

---

### Schritt-für-Schritt-Anleitung

#### Schritt 1 – Support-Domäne wählen (5 Min)

Einige sich im Team auf eine konkrete Domäne:

| Domäne | FAQ-Beispiele | Specialist-Beispiele |
|--------|--------------|---------------------|
| 🛍️ Online-Shop | Lieferzeit, Rückgabe, Preise | Doppelabbuchung, technischer Defekt |
| 💻 Software-Support | Passwort, Update, Installation | Datenverlust, komplexer Bug |
| 🏥 Arztpraxis | Öffnungszeiten, Termine, Kosten | Symptome, Medikamentenfragen |
| 🏢 Interner IT-Support | WLAN-Passwort, VPN-Verbindung | Systemabsturz, Datensicherung |
| 🎓 Kurs-Support | Zugangsdaten, Materialien | Technische Probleme, inhaltliche Fragen |

Tragt eure Domäne ein: **Unsere Domäne: ___________________________**

---

#### Schritt 2 – FAQ-Datenbank füllen (15 Min)

Das ist der **wichtigste Schritt** für das Support-Team! Ohne eine gute FAQ-Datenbank
kann der FAQ-Agent nicht sinnvoll antworten.

Öffne die **Tools**-Zelle und erweitere `FAQ_DATABASE` mit **mindestens 8 echten Einträgen** aus eurer Domäne:

```python
FAQ_DATABASE = {
    # ⚠️ TODO: Ersetze alle Platzhalter durch echte Einträge
    "schluessel1": "Klare, vollständige Antwort auf die häufigste Frage.",
    "schluessel2": "Antwort mit konkreten Schritten oder Links.",
    "schluessel3": "...",
    # ... mindestens 8 Einträge
}
```

**Tipp:** Nutze kurze, prägnante Schlüsselwörter, die der FAQ-Agent leicht finden kann
(z.B. `"passwort"`, `"lieferung"`, `"kosten"` – nicht `"wie_kann_ich_mein_passwort_aendern"`).

---

#### Schritt 3 – Specialist-Tools anpassen (10 Min)

Passe `analyze_problem` und `propose_solution` an eure Domäne an:

```python
@tool
def analyze_problem(beschreibung: str) -> str:
    """..."""
    # ⚠️ TODO: Welche Kategorien gibt es in eurer Domäne?
    return (
        f"Analyse für: '{beschreibung}'\n"
        "Kategorie: [EURER KATEGORIE z.B. 'Technisch', 'Abrechnung', 'Logistik']\n"
        "Dringlichkeit: [hoch/mittel/niedrig]\n"
        "Benötigte Info: [was braucht ihr noch?]"
    )
```

---

#### Schritt 4 – Routing-Logik im Supervisor anpassen (10 Min)

Das ist das **Herzstück** des Support-Teams: Der Supervisor muss entscheiden,
ob eine Anfrage einfach (→ FAQ) oder komplex (→ Specialist) ist.

Öffne die **Supervisor-Prompt**-Zelle und definiere klare Routing-Regeln:

```python
SUPERVISOR_PROMPT = (
    ...
    "<Routing-Regeln>\n"
    # ⚠️ TODO: Definiere konkret wann FAQ und wann Specialist
    "→ faq:        [konkrete Beispiele aus eurer Domäne]\n"
    "→ specialist: [konkrete Beispiele aus eurer Domäne]\n"
    "→ Eskalation: faq antwortet 'nicht gefunden' → specialist\n"
    "</Routing-Regeln>\n"
    ...
)
```

**Tipp:** Zu vage Routing-Regeln führen dazu, dass der Supervisor immer denselben
Agenten wählt. Seid so konkret wie möglich!

---

#### Schritt 5 – Mit zwei Testfällen prüfen (10 Min)

Das Template hat bereits **zwei Testläufe**: einen für einfache, einen für komplexe Anfragen.

1. Passe beide Anfragen an eure Domäne an
2. Führe alle Zellen von oben nach unten aus
3. Prüfe: Wurde die einfache Anfrage wirklich zum FAQ-Agenten gerouted?
4. Prüfe: Wurde die komplexe Anfrage wirklich zum Specialist gerouted?
5. Hake die MVP-Checkliste ab

---

### Mini-Demo am Ende (3 Min pro Team)

Zeigt euren Mitstreitern:
1. **Einfache Anfrage** → FAQ-Agent antwortet direkt (schnell, günstig)
2. **Komplexe Anfrage** → Specialist übernimmt mit Analyse
3. **LangSmith-Trace** → Der Routing-Entscheid des Supervisors ist sichtbar

---

### Bonus-Aufgaben (wenn das MVP läuft)

**Bonus 1 – Eskalations-Tracking**
Erweitere den State um ein `eskalationen`-Feld, das zählt wie oft ein Fall
vom FAQ-Agenten zum Specialist eskaliert wurde:

```python
class AgentState(TypedDict):
    ...
    eskalationen: int   # ← NEU: Eskalations-Zähler
```

**Bonus 2 – FAQ-Lernschleife**
Wenn der FAQ-Agent „nicht gefunden" zurückgibt und der Specialist erfolgreich
antwortet: Füge die Frage automatisch zur `FAQ_DATABASE` hinzu.

```python
@tool
def add_to_faq(schluessel: str, antwort: str) -> str:
    """Fügt eine neue Frage+Antwort zur FAQ-Datenbank hinzu."""
    FAQ_DATABASE[schluessel] = antwort
    return f"FAQ erweitert: '{schluessel}' gespeichert."
```

**Bonus 3 – Ticket-Erstellung**
Wenn der Specialist eine Frage nicht lösen kann, soll er automatisch ein
Support-Ticket (als Dictionary oder JSON-Datei) erstellen.

**Bonus 4 – Human in the Loop**
Baue einen Genehmigungsschritt ein: Bevor der Specialist eine kritische Aktion
empfiehlt (z.B. Rückerstattung), muss ein Mensch bestätigen (→ *M17*).
